In [1]:
import pandas as pd

In [2]:
import pandas as pd

season = "2627"
praefix = season[:2]  
leagues = {
    "pl": "E0",  
    "de": "D1",  
    "es": "SP1",  
    "it": "I1",  
    "fr": "F1", 
}
urls = {
    name: f"https://football-data.co.uk/mmz4281/{season}/{code}.csv"
    for name, code in leagues.items()
}

In [3]:
import numpy as np


In [4]:
def snake_case(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


In [5]:
column_mapping = {
    "hometeam": "home_team",
    "awayteam": "away_team",
    "fthg": "full_time_home_goals",
    "ftag": "full_time_away_goals",
    "ftr": "full_time_result",
    "hthg": "half_time_home_goals",
    "htag": "half_time_away_goals",
    "htr": "half_time_result",
    "hs": "home_shots",
    "as": "away_shots",
    "hst": "home_shots_on_target",
    "ast": "away_shots_on_target"
}



In [6]:
columns_for_standings = [
    "date",
    "home_team",
    "away_team",
    "full_time_home_goals",
    "full_time_away_goals",
    "full_time_result",
]

In [7]:
league_data = {}

for name, code in leagues.items():
    league_data[name] = {}

    full = pd.read_csv(urls[name], encoding="utf-8-sig")
    full = snake_case(full)
    full = full.rename(columns=column_mapping)

    league_data[name]["full"] = full
    league_data[name]["matches"] = full[columns_for_standings].copy()

    matches = league_data[name]["matches"]

    # Combine home and away teams, remove duplicates, then sort them.
    league_data[name]["teams"] = sorted(
        set(matches["home_team"]) | set(matches["away_team"])
    )

    league_data[name]["team_matches"] = {}

    for team in league_data[name]["teams"]:
        team_matches = matches[
            (matches["home_team"] == team)
            | (matches["away_team"] == team)
        ].copy()

        league_data[name]["team_matches"][team] = team_matches


In [8]:
for name in leagues:
    standings_rows = []

    for team in league_data[name]["teams"]:
        club_df = league_data[name]["team_matches"][team]

        
        played = club_df.dropna(
            subset=["full_time_home_goals", "full_time_away_goals"]
        )

        home_games = played[played["home_team"] == team]
        away_games = played[played["away_team"] == team]

        home_wins = (
            home_games["full_time_home_goals"]
            > home_games["full_time_away_goals"]
        ).sum()
        away_wins = (
            away_games["full_time_away_goals"]
            > away_games["full_time_home_goals"]
        ).sum()
        wins = home_wins + away_wins

        home_draws = (
            home_games["full_time_home_goals"]
            == home_games["full_time_away_goals"]
        ).sum()
        away_draws = (
            away_games["full_time_away_goals"]
            == away_games["full_time_home_goals"]
        ).sum()
        draws = home_draws + away_draws

        home_losses = (
            home_games["full_time_home_goals"]
            < home_games["full_time_away_goals"]
        ).sum()
        away_losses = (
            away_games["full_time_away_goals"]
            < away_games["full_time_home_goals"]
        ).sum()
        losses = home_losses + away_losses

        goals_for = (
            home_games["full_time_home_goals"].sum()
            + away_games["full_time_away_goals"].sum()
        )
        goals_against = (
            home_games["full_time_away_goals"].sum()
            + away_games["full_time_home_goals"].sum()
        )
        goal_difference = goals_for - goals_against
        points = 3 * wins + draws

        standings_rows.append({
            "Team": team,
            "Played": len(played),
            "W": wins,
            "D": draws,
            "L": losses,
            "GF": goals_for,
            "GA": goals_against,
            "GD": goal_difference,
            "Points": points,
        })

    league_data[name]["standings"] = (
        pd.DataFrame(standings_rows)
        .sort_values(
            ["Points", "GD", "GF"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


In [9]:
file_path = "football_standings.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    for name in leagues:
        table = league_data[name]["standings"].copy()
        table.index = range(1, len(table) + 1)

        table.to_excel(
            writer,
            sheet_name=name.upper(),
            index_label="Pos"
        )

In [ ]:
for name in leagues:
    standings_rows = []

    for team in league_data[name]["teams"]:
        club_df = league_data[name]["team_matches"][team]

        
        played = club_df.dropna(
            subset=["full_time_home_goals", "full_time_away_goals"]
        )

        home_games = played[played["home_team"] == team]
        away_games = played[played["away_team"] == team]

        home_wins = (
            home_games["full_time_home_goals"]
            > home_games["full_time_away_goals"]
        ).sum()
        away_wins = (
            away_games["full_time_away_goals"]
            > away_games["full_time_home_goals"]
        ).sum()
        wins = home_wins + away_wins

        home_draws = (
            home_games["full_time_home_goals"]
            == home_games["full_time_away_goals"]
        ).sum()
        away_draws = (
            away_games["full_time_away_goals"]
            == away_games["full_time_home_goals"]
        ).sum()
        draws = home_draws + away_draws

        home_losses = (
            home_games["full_time_home_goals"]
            < home_games["full_time_away_goals"]
        ).sum()
        away_losses = (
            away_games["full_time_away_goals"]
            < away_games["full_time_home_goals"]
        ).sum()
        losses = home_losses + away_losses

        goals_for = (
            home_games["full_time_home_goals"].sum()
            + away_games["full_time_away_goals"].sum()
        )
        goals_against = (
            home_games["full_time_away_goals"].sum()
            + away_games["full_time_home_goals"].sum()
        )
        goal_difference = goals_for - goals_against
        points = 3 * wins + draws

        standings_rows.append({
            "Team": team,
            "Played": len(played),
            "W": wins,
            "D": draws,
            "L": losses,
            "GF": goals_for,
            "GA": goals_against,
            "GD": goal_difference,
            "Points": points,
        })

    league_data[name]["standings"] = (
        pd.DataFrame(standings_rows)
        .sort_values(
            ["Points", "GD", "GF"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


,Team,Played,W,D,L,GF,GA,GD,Points
1,Arsenal,10,9,0,1,20,6,14,27
2,Brighton,10,7,1,2,30,12,18,22
3,Man City,10,7,0,3,22,12,10,21
4,Liverpool,10,5,2,2,8,8,0,17
5,Chelsea,10,5,1,4,26,16,10,16
6,Bournemouth,10,4,2,3,14,20,-6,14
7,Everton,10,4,1,2,16,10,6,13
8,Leeds,10,3,3,2,14,10,4,12
9,Newcastle,10,3,2,4,12,18,-6,11
10,Aston Villa,10,3,2,3,12,20,-8,11


In [17]:
standings_dfs = {}
for name in leagues:
    standings_dfs[name] = league_data[name]["standings"]


In [18]:
full

,div,date,time,home_team,away_team,full_time_home_goals,full_time_away_goals,full_time_result,half_time_home_goals,half_time_away_goals,...,bfec<2.5,ahch,b365cahh,b365caha,maxcahh,maxcaha,avgcahh,avgcaha,bfecahh,bfecaha
0,F1,21/08/2026,19:45,Marseille,Strasbourg,4,0,H,0,0,...,2.72,-0.75,1.98,1.88,1.98,1.90,1.90,1.84,1.99,2.00
1,F1,22/08/2026,16:15,Lens,Auxerre,5,2,H,1,0,...,2.28,-1.25,1.93,1.93,1.93,1.93,1.90,1.84,2.05,1.90
2,F1,22/08/2026,19:45,Le Mans,Brest,2,2,D,1,1,...,1.95,0.00,1.80,2.05,1.80,2.05,1.78,1.97,1.90,2.10
3,F1,22/08/2026,19:45,Nice,Lorient,0,0,D,0,0,...,1.82,-0.25,2.00,1.85,2.00,1.85,1.97,1.78,2.12,1.87
4,F1,22/08/2026,19:45,Toulouse,Lyon,0,2,A,0,0,...,2.00,-0.25,1.93,1.93,1.93,1.93,1.88,1.86,2.01,1.97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,F1,08/11/2026,16:15,Angers,Nice,0,2,A,0,1,...,2.18,0.50,2.00,1.85,2.00,1.85,1.93,1.81,2.06,1.92
86,F1,08/11/2026,16:15,Brest,Lyon,2,0,H,1,0,...,2.06,0.00,2.00,1.85,2.00,1.85,1.95,1.79,2.10,1.89
87,F1,08/11/2026,16:15,Le Havre,Lorient,0,2,A,0,1,...,1.81,0.25,1.80,2.05,1.80,2.05,1.77,1.99,1.87,2.14
88,F1,08/11/2026,16:15,Le Mans,Monaco,1,1,D,0,1,...,2.12,0.75,2.05,1.75,2.05,1.80,2.02,1.73,2.15,1.86
